#compared to latest gpt models this will be basic

In [2]:
#Self attention-generates the output with respect to the context of all the input tokens....selctive access
#Tokens are assigned importance based on weights(attention_scores)
#attention scores=unnormalized attention wts()

import torch

inputs=torch.tensor(
    [
        [0.43,0.15,0.89],#Your
        [0.55,0.87,0.66],#journey
        [0.57,0.85,0.64],#starts
        [0.22,0.58,0.33],#with
        [0.77,0.25,0.10],#one
        [0.05,0.80,0.55] #step
    ]
)

In [3]:
input_query=inputs[1]
input_query


tensor([0.5500, 0.8700, 0.6600])

In [4]:
#What happens
0.55*0.43+0.87*0.15+0.66*0.89

0.9544

In [5]:
res=0.
for j,ele in enumerate(inputs[1]):
    res+=inputs[0][j]*input_query[j]
res

tensor(0.9544)

In [6]:
r=[]
for i in range(0,6):
    res=0.
    for j,ele in enumerate(inputs[i]):
        res+=inputs[i][j]*input_query[j]
        r.append(res)
r

[tensor(0.9544),
 tensor(0.9544),
 tensor(0.9544),
 tensor(1.4950),
 tensor(1.4950),
 tensor(1.4950),
 tensor(1.4754),
 tensor(1.4754),
 tensor(1.4754),
 tensor(0.8434),
 tensor(0.8434),
 tensor(0.8434),
 tensor(0.7070),
 tensor(0.7070),
 tensor(0.7070),
 tensor(1.0865),
 tensor(1.0865),
 tensor(1.0865)]

In [7]:
i=3
for j,ele in enumerate(inputs[i]):
    res=torch.dot(input_query,inputs[0])
res

tensor(0.9544)

In [8]:
query=inputs[1]
attn_scores_2=torch.empty(inputs.shape[0]) #tensor([0.,0.,0.,0.])
for i,x_i in enumerate(inputs):
    attn_scores_2[i]=torch.dot(x_i,query)     #score[i]=x[i]*query
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [9]:
attn_wt2_temp=attn_scores_2/attn_scores_2.sum()
attn_wt2_temp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [10]:
attn_wt2_temp.sum()

tensor(1.0000)

In [11]:
def softmax_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)
softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [12]:
torch.softmax(attn_scores_2,dim=0)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [13]:
query=inputs[1]

context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+=attn_wt2_temp[i]*x_i
print(context_vec_2)

tensor([0.4355, 0.6451, 0.5680])


In [14]:
for i,x_i in enumerate(inputs):
    print("Index:",i,x_i)

Index: 0 tensor([0.4300, 0.1500, 0.8900])
Index: 1 tensor([0.5500, 0.8700, 0.6600])
Index: 2 tensor([0.5700, 0.8500, 0.6400])
Index: 3 tensor([0.2200, 0.5800, 0.3300])
Index: 4 tensor([0.7700, 0.2500, 0.1000])
Index: 5 tensor([0.0500, 0.8000, 0.5500])


Self attention without Trainable Weights

In [15]:
# query=inputs[1]
attn_scores=torch.empty(6,6)
for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j]=torch.dot(x_i,x_j)
print(attn_scores) #Not normalized yet

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [16]:
#instead of for loops we can do matrix multiplication that can be efficient
attn_scores=inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [17]:
attn_weights=torch.softmax(attn_scores,dim=1)#sum of rows=1
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [18]:
all_context_vecs=attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

Computing attention weights step by step with trainable weights

In [19]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [20]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2


In [21]:
torch.manual_seed(123)

W_query=torch.nn.Parameter(torch.rand(d_in,d_out))#d-dimension
W_key=torch.nn.Parameter(torch.rand(d_in,d_out))
W_value=torch.nn.Parameter(torch.rand(d_in,d_out))
query_2=x_2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [22]:
keys=inputs @ W_key
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [23]:
value=inputs @W_value
value
value.shape

torch.Size([6, 2])

In [24]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [25]:
attn_scores_2=query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [28]:
#Compute normalized attention weights
d_k=keys.shape[1]
attn_weights_2=torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [29]:
context_vector_2=attn_weights_2 @ value
context_vector_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

## Implementing a Self Attention Class

In [32]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()#for inheritence
        self.W_query=nn.Parameter(torch.rand(d_in,d_out))
        self.W_key=nn.Parameter(torch.rand(d_in,d_out))
        self.W_value=nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,x):
        queries=inputs @ W_query
        keys=inputs @W_key
        values=inputs @W_value

        attn_scores=queries @ keys.T
        attn_weights=torch.softmax(attn_scores/d_k**0.5,dim=-1)
        context_vec=attn_weights @ values
        return context_vec
torch.manual_seed(123)
sav1=SelfAttention_v1(d_in,d_out)
sav1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

Version 2

In [35]:
class SelfAttention_v2(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias=False):
        super().__init__()#for inheritence
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)  #Linear replaces torch rand
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,x):
        queries=self.W_query(inputs)
        keys=self.W_key(inputs)
        values=self.W_value(inputs)

        attn_scores=queries @ keys.T
        attn_weights=torch.softmax(attn_scores/d_k**0.5,dim=-1)
        context_vec=attn_weights @ values
        return context_vec

# Applying a Causal Attention Mask(Hiding Future Words)

Your journey starts with one step
Your->journey->rest should be hidden

In [36]:
sa_v2=SelfAttention_v2(d_in,d_out)
queries=sa_v2.W_query(inputs)
keys=sa_v2.W_key(inputs)
values=sa_v2.W_value(inputs)

attn_scores=queries @ keys.T
attn_weights=torch.softmax(attn_scores/d_k**0.5,dim=-1)

attn_weights




tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)

In [40]:
context_length=attn_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [43]:
masked_simple=attn_weights*mask_simple
masked_simple

tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<MulBackward0>)

In [ ]:
row_sum=masked_simple.sum(dim=-1,keepdim=True)
masked_simple_norm=masked_simple/row_sum
print(masked_simple_norm)#Normalized...sum up to 1 in each row

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)


### Simpler way of entire process

In [ ]:
# Mask first then softmax i.e 0s replaced by -inf
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked=attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [47]:
torch.exp(torch.tensor(-999999999))

tensor(0.)

In [48]:
torch.exp(torch.tensor(float("-inf")))

tensor(0.)

In [49]:
attn_weights=torch.softmax(masked/d_k**0.5,dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)

### Masking Additional Attention Weights using Dropout

In [60]:
#Dropout random places
#torch.nn.Dropout(0.5)#drop 50% of the positions

torch.manual_seed(123)
layer=torch.nn.Dropout(0.5)



In [61]:
example=torch.ones(6,6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [62]:
layer(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [63]:
# dropout_rate=0.5
# 1/(1-dropout_rate)
layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7160, 0.7181, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5167, 0.5147, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4101, 0.0000],
        [0.2815, 0.3430, 0.0000, 0.3434, 0.3516, 0.3368]],
       grad_fn=<MulBackward0>)

### Implementing a compact causal self-attention class

In [65]:
# inputs #each inp token is 3d vecotr now we stack 2 inputs on top of each other
batch=torch.stack((inputs,inputs),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [70]:
class CausalAttentionV2(nn.Module):
    def __init__(self,d_in,d_out,dropout,context_length,qkv_bias=False):
        super().__init__()#for inheritence
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)  #Linear replaces torch rand
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout=torch.nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))#Optionally train on gpu,pytorch doesnt move over any tensors so we have to register those tensors



    def forward(self,x):
        b,num_tokens,d_in=x.shape
        queries=self.W_query(x)
        keys=self.W_key(x)
        values=self.W_value(x)

        attn_scores=queries @ keys.transpose(1,2) #Changed Transpose
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens,:num_tokens],-torch.inf
        )
        attn_weights=torch.softmax(attn_scores/keys.shape[1]**0.5,dim=-1)
        context_vec=attn_weights @ values
        return context_vec

In [75]:
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [67]:
batch.shape

torch.Size([2, 6, 3])

In [74]:
torch.manual_seed(123)
context_length=batch.shape[1]
dropout=0.0
ca=CausalAttentionV2(d_in,d_out,dropout,context_length)
ca(batch) #6 rows but 2 columns

tensor([[[-0.4519,  0.2216],
         [-0.5856,  0.0087],
         [-0.6284, -0.0607],
         [-0.5664, -0.0832],
         [-0.5511, -0.0978],
         [-0.5290, -0.1073]],

        [[-0.4519,  0.2216],
         [-0.5856,  0.0087],
         [-0.6284, -0.0607],
         [-0.5664, -0.0832],
         [-0.5511, -0.0978],
         [-0.5290, -0.1073]]], grad_fn=<UnsafeViewBackward0>)

# Extending Single Head to Multi Head Attention

### Stacking multiple single head Attention Layers

In [76]:
class MultiHead_AttentionWrapper(nn.Module):
    def __init__(self,d_in,d_out,dropout,context_length,num_heads=2,qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList(
        [CausalAttentionV2(d_in,d_out,dropout,context_length,qkv_bias) for _ in range(num_heads)]
        )
    
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)
torch.manual_seed(123)
context_length=batch.shape[1]
dropout=0.0
d_in,d_out=batch.shape[2],batch.shape[0]
mha=MultiHead_AttentionWrapper(d_in,d_out,dropout,context_length)

In [77]:
mha(batch)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5856,  0.0087,  0.5840,  0.3158],
         [-0.6284, -0.0607,  0.6161,  0.3779],
         [-0.5664, -0.0832,  0.5468,  0.3557],
         [-0.5511, -0.0978,  0.5317,  0.3416],
         [-0.5290, -0.1073,  0.5072,  0.3465]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5856,  0.0087,  0.5840,  0.3158],
         [-0.6284, -0.0607,  0.6161,  0.3779],
         [-0.5664, -0.0832,  0.5468,  0.3557],
         [-0.5511, -0.0978,  0.5317,  0.3416],
         [-0.5290, -0.1073,  0.5072,  0.3465]]], grad_fn=<CatBackward0>)

### Implementing Multi head Attention with Weight Splits

In [81]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads==0),\
            "d_out must be divisible by num_heads"
        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj=nn.Linear(d_out,d_out)
        self.dropout=nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length),diagonal=1)
        )
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        keys=self.W_key(x)
        queries=self.W_query(x)
        values=self.W_value(x)
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b,num_tokens,self.num_heads,self.head_dim)
        queries=queries.view(b,num_tokens,self.num_heads,self.head_dim)

        keys=keys.transpose(1,2)
        queries=queries.transpose(1,2)
        values=values.transpose(1,2)

        attn_scores = queries @ keys.transpose(2,3)

        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
        attn_scores.masked_fill_(mask_bool,-torch.inf)

        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weights=self.dropout(attn_weights)

        context_vec=(attn_weights @ values).transpose(1,2)
        context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
        context_vec=self.out_proj(context_vec)

        return context_vec
    
torch.manual_seed(123)

batch_size,context_length,d_in=batch.shape
d_out=4
mha=MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads=2)
context_vecs=mha(batch)

print("Context_vectors are: \n",context_vecs)


        

Context_vectors are: 
 tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)


In [ ]:
#view and reshape are similar---reshape creates a copy view does not
